# Boundary-Aware Inpainting — Full Pipeline

**Before running:**
1. Runtime → Change runtime type → **A100 GPU** (Colab Pro/Pro+)
2. Mount Google Drive (Cell 1) — saves checkpoints so you don't lose them on disconnect
3. Run cells top to bottom

**Time estimate (A100, production config):**
- Smoke test (5 epochs, 0.5% data): ~15 min
- Full pipeline (12 epochs, 100% data, 6 configs): ~29 hours
- Use `--skip_phases123 --skip_multiseed --skip_convnext` to save ~8 hours

In [ ]:
# ── Cell 1: Mount Google Drive (REQUIRED for checkpoint persistence) ───────────
from google.colab import drive
drive.mount('/content/drive')

# All checkpoints and results will be saved here
# so they survive Colab disconnects
DRIVE_ROOT = '/content/drive/MyDrive/boundary_inpainting'

import os
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/checkpoints', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/results', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/datasets', exist_ok=True)

print('Drive mounted. Root:', DRIVE_ROOT)

In [ ]:
# ── Cell 2: Install dependencies ───────────────────────────────────────────────
import subprocess, sys

pkgs = [
    'lpips',          # perceptual metric
    'einops',         # transformer bottleneck
    'scikit-image',   # SSIM
    'clean-fid',      # FID computation
    'timm',           # ConvNeXt backbone
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs, check=True)
print('Dependencies installed.')

In [ ]:
# ── Cell 3: Upload and extract code ────────────────────────────────────────────
# Option A: Upload zip from local machine
from google.colab import files
print('Upload your boundary_inpainting_final.zip file...')
uploaded = files.upload()  # select the zip

import zipfile, os
zip_name = list(uploaded.keys())[0]
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

# Find the extracted folder
CODE_DIR = '/content/boundary_inpainting_final'
os.chdir(CODE_DIR)
sys.path.insert(0, CODE_DIR)
print('Code extracted to:', CODE_DIR)
print('Files:', os.listdir(CODE_DIR))

In [ ]:
# ── Cell 4: Symlink checkpoints/results to Drive (persistence) ────────────────
import os

CODE_DIR  = '/content/boundary_inpainting_final'
DRIVE_ROOT = '/content/drive/MyDrive/boundary_inpainting'

def _link(src, dst):
    os.makedirs(src, exist_ok=True)
    if os.path.islink(dst):
        os.unlink(dst)
    elif os.path.isdir(dst):
        import shutil; shutil.rmtree(dst)
    os.symlink(src, dst)
    print(f'  {dst} → {src}')

print('Symlinking to Drive:')
_link(f'{DRIVE_ROOT}/checkpoints', f'{CODE_DIR}/checkpoints')
_link(f'{DRIVE_ROOT}/results',     f'{CODE_DIR}/results')
print('Done. Checkpoints will persist across Colab sessions.')

In [ ]:
# ── Cell 5: Download / restore datasets ────────────────────────────────────────
import os, sys
CODE_DIR   = '/content/boundary_inpainting_final'
DRIVE_ROOT = '/content/drive/MyDrive/boundary_inpainting'
sys.path.insert(0, CODE_DIR)
os.chdir(CODE_DIR)

from data.download import download_all

# drive_cache: if datasets were previously downloaded and saved to Drive,
#              this restores them (~5 min) instead of re-downloading (~45 min)
download_all(
    base_dir='./datasets',
    drive_cache=f'{DRIVE_ROOT}/datasets',
)

# After first successful download, backup datasets to Drive for future sessions
import shutil
drive_ds = f'{DRIVE_ROOT}/datasets'
if not os.path.exists(f'{drive_ds}/.backed_up'):
    print('Backing up datasets to Drive (one-time, ~2-3 min)...')
    shutil.copytree('./datasets', drive_ds, dirs_exist_ok=True)
    open(f'{drive_ds}/.backed_up', 'w').close()
    print('Backup done.')
else:
    print('Datasets already backed up on Drive.')

In [ ]:
# ── Cell 6: GPU / environment check ────────────────────────────────────────────
import torch
print(f'PyTorch : {torch.__version__}')
print(f'CUDA    : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU     : {torch.cuda.get_device_name(0)}')
    print(f'VRAM    : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    if vram < 15:
        print('WARNING: Less than 15 GB VRAM. Reduce batch_size to 16 or 8.')
    elif vram >= 35:
        print('A100 detected. batch_size=48 is optimal.')
    else:
        print('T4/V100 detected. Consider batch_size=16-24.')

In [ ]:
# ── Cell 7: SMOKE TEST — run this first to verify pipeline works ───────────────
# Runs 5 epochs on ~180 images. Should complete in ~15 min on A100.
# If this passes without errors, run Cell 8 for the full pipeline.

import subprocess, sys, os
CODE_DIR = '/content/boundary_inpainting_final'
os.chdir(CODE_DIR)

result = subprocess.run(
    [
        sys.executable, 'main.py',
        '--profile',      'smoke',
        '--places_dir',   './datasets/places365',
        '--celeba_dir',   './datasets/celeba_hq',
        '--dtd_dir',      './datasets/dtd',
        '--skip_phases123',
        '--skip_multiseed',
        '--skip_convnext',
        '--num_workers',  '2',
    ],
    cwd=CODE_DIR,
)
print('\nSmoke test exit code:', result.returncode)
if result.returncode == 0:
    print('SMOKE TEST PASSED — run Cell 8 for full pipeline.')
else:
    print('SMOKE TEST FAILED — check errors above.')

In [ ]:
# ── Cell 8: FULL PIPELINE (production) ────────────────────────────────────────
#
# Estimated time on A100:
#   With --skip_phases123 --skip_multiseed --skip_convnext : ~17 hours (6 ablation configs × 12 epochs)
#   Full pipeline (all phases, multi-seed, ConvNeXt)        : ~29 hours
#
# Colab Pro+ gives 24h sessions. With skip flags: fits in one session.
# If disconnected: re-run this cell — it auto-resumes from _resume.pt checkpoints.
#
# IMPORTANT: This cell will run for many hours.
# Keep the browser tab open or use Colab's scheduled runtime feature.

import subprocess, sys, os
CODE_DIR = '/content/boundary_inpainting_final'
os.chdir(CODE_DIR)

result = subprocess.run(
    [
        sys.executable, 'main.py',
        '--profile',      'production',
        '--places_dir',   './datasets/places365',
        '--celeba_dir',   './datasets/celeba_hq',
        '--dtd_dir',      './datasets/dtd',
        '--num_workers',  '4',
        # ── Recommended flags to fit in 24h session ──
        '--skip_phases123',   # use resnet34 + attn_skip + gated_conv (proven best)
        '--skip_multiseed',   # skip seeds 1 and 2 (seed 42 still runs)
        '--skip_convnext',    # skip ConvNeXt runs to save ~4h
        # ── Remove flags above for complete paper run ──
    ],
    cwd=CODE_DIR,
)
print('Pipeline exit code:', result.returncode)

In [ ]:
# ── Cell 9: Download results ───────────────────────────────────────────────────
# Results are already saved to Drive. Also download locally as backup.

import shutil, os
from google.colab import files

CODE_DIR = '/content/boundary_inpainting_final'

# Zip results
results_zip = '/content/results.zip'
shutil.make_archive('/content/results', 'zip', f'{CODE_DIR}/results')
print(f'Results zipped: {os.path.getsize(results_zip)/1e6:.1f} MB')
files.download(results_zip)

# Zip checkpoints (large — only download if needed)
# shutil.make_archive('/content/checkpoints', 'zip', f'{CODE_DIR}/checkpoints')
# files.download('/content/checkpoints.zip')

In [ ]:
# ── Cell 10: Extend training (optional) ───────────────────────────────────────
# Run this if you want to extend specific configs to more epochs.
# Uses warm-restart from _ema.pt with reduced LR.

import subprocess, sys, os
CODE_DIR = '/content/boundary_inpainting_final'
os.chdir(CODE_DIR)

result = subprocess.run(
    [
        sys.executable, 'main.py',
        '--profile',        'production',
        '--places_dir',     './datasets/places365',
        '--celeba_dir',     './datasets/celeba_hq',
        '--dtd_dir',        './datasets/dtd',
        '--num_workers',    '4',
        '--skip_phases123',
        '--skip_multiseed',
        '--skip_convnext',
        '--extend_epochs',  '20',   # extend to 20 total epochs
        '--extend_lr_frac', '0.2',  # use 20% of original LR
        '--extend_configs', 'L0_base,L3_spectral_only,L3c_adaptive_spectral',
    ],
    cwd=CODE_DIR,
)
print('Extend exit code:', result.returncode)